# Users Dimension — Gold Layer (`company_db.gold.users_dim`)

This notebook builds and maintains the **Users Dimension** table in the Gold layer of the Company data warehouse. It runs as part of the batch orchestration pipeline.

---

## What This Notebook Does

1. Extracts and deduplicates user records from the Silver layer
2. Enriches them with date-part columns (year, quarter, month, day, hour)
3. Assigns or preserves surrogate keys (`user_dim_key`)
4. Merges the result into the Gold dimension table (insert new users, update existing ones)

---

## Source Tables (Silver Layer)

| Table | Description |
|---|---|
| `company_db.silver.users` | All registered platform users with billing contact details |
| `company_db.silver.registeredusers` | Auction-level registrations linking users to specific auctions |

## Target Table (Gold Layer)

`company_db.gold.users_dim`  
Stored in Azure Data Lake Storage Gen2: `abfss://auction@company.dfs.core.windows.net/gold/users_dim`

---

## Load Strategy (controlled by the `flag` widget)

| `flag` value | Behavior |
|---|---|
| `0` | **Initial / Full Load** — surrogate key sequence starts at 1 |
| `1` | **Incremental Load** — surrogate key sequence continues from current `MAX(user_dim_key)` |

---

## Column Reference

| Column | Type | Description |
|---|---|---|
| `user_dim_key` | BIGINT | **Surrogate key** — auto-incrementing warehouse identifier |
| `user_id` | INT | Source system user identifier |
| `full_name` | STRING | User's full name |
| `phone_number` | STRING | Billing phone number |
| `email` | STRING | Email address |
| `address` | STRING | Billing address line 1 |
| `address_2` | STRING | Billing address line 2 |
| `city` | STRING | Billing city |
| `province` | STRING | Billing state / province |
| `country` | STRING | Billing country |
| `registered_date` | TIMESTAMP | Account creation timestamp |
| `registered_year` | INT | Year extracted from `registered_date` |
| `registered_quarter` | INT | Quarter (1–4) extracted from `registered_date` |
| `registered_month` | STRING | Month name (e.g. `January`) |
| `registered_day` | INT | Day of month |
| `registered_hour` | INT | Hour of day (0–23) |
| `registered_day_name` | STRING | Day of week name (e.g. `Monday`) |
| `updated_at` | TIMESTAMP | Timestamp of the last MERGE update |
| `created_at` | TIMESTAMP | Timestamp when the row was first inserted |

In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
from pyspark.sql import functions as F

---
## Step 1 · Extract Users from Silver Layer

Joins `company_db.silver.users` with `company_db.silver.registeredusers` on `user_id` (LEFT JOIN to retain users without auction registrations). Renames billing columns to friendlier dimension names and deduplicates on `user_id`, keeping the earliest `date_created` record per user.

In [ ]:
users_table = spark.sql(f"""
with users as 
(SELECT 
users_table.*,registered_user_table.auction_id
from
{catalog}.silver.users as users_table
left join {catalog}.silver.registeredusers as registered_user_table
on users_table.user_id = registered_user_table.user_id
order by users_table.date_created
)
select
distinct(user_id),
full_name,
billing_phone AS phone_number,
email_address as email,
billing_address_1 as address,
billing_address_2 as address_2,
billing_city as city,
billing_state as province,
billing_country as country,
date_created as registered_date
FROM 
users;
"""
)


In [ ]:
users_table = users_table.withColumns({'registered_year': F.year(F.col("registered_date")),'registered_quarter': F.quarter(F.col("registered_date")), 'registered_month': F.monthname(F.col("registered_date")),'registered_day': F.day(F.col("registered_date")),'registered_day_name': F.dayname(F.col("registered_date")),'registered_hour': F.hour(F.col("registered_date"))})

---
## Step 2 · Date Dimension Enrichment

Breaks `registered_date` into individual date-part columns to support slicing and filtering in dashboards and reports without needing date functions at query time.

| New Column | Derived From | Example |
|---|---|---|
| `registered_year` | `YEAR(registered_date)` | `2023` |
| `registered_quarter` | `QUARTER(registered_date)` | `2` |
| `registered_month` | `MONTHNAME(registered_date)` | `June` |
| `registered_day` | `DAY(registered_date)` | `15` |
| `registered_day_name` | `DAYNAME(registered_date)` | `Thursday` |
| `registered_hour` | `HOUR(registered_date)` | `14` |

In [ ]:
# flag =  dbutils.widgets.text('flag','0')

In [ ]:
# # flag = dbutils.widgets.get('flag')
# flag

---
## Step 3 · Load Control Flag & Gold Table Initialization

**`flag` widget** controls the load mode for this run:
- `flag = 0` — **Initial / Full Load.** Surrogate key counter starts at 1. Use this on the first ever run or after a full table reset.
- `flag = 1` — **Incremental Load.** Surrogate key counter continues from `MAX(user_dim_key)` in the existing table. Use this for every scheduled run.

**`CREATE TABLE IF NOT EXISTS`** ensures the Gold Delta table exists before any data is written. If the table already exists this statement is a no-op — no data is affected.

In [ ]:
spark.sql(
f"""
CREATE TABLE IF NOT EXISTS {catalog}.gold.users_dim
(
user_dim_key BIGINT,
user_id integer,
full_name string,
phone_number string,
email string,
address string,
address_2 string,
city string,
province string,
country string,
registered_date timestamp,
registered_year integer,
registered_quarter integer,
registered_month string,
registered_day integer,
registered_hour integer,
registered_day_name string,
updated_at timestamp,
created_at timestamp)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/gold/users_dim';""")

In [ ]:
users = spark.table(f'{catalog}.gold.users_dim')

---
## Step 4 · Identify New vs Existing Users

Loads the current `company_db.gold.users_dim` and LEFT JOINs it against the incoming Silver users on `user_id`.

After the join, records are split into two groups:

| Group | Condition | Next Action |
|---|---|---|
| `new_users` | `user_dim_key IS NULL` | Needs a surrogate key assigned, then INSERTed |
| `old_users` | `user_dim_key IS NOT NULL` | Surrogate key already exists, will be UPDATEd via MERGE |

In [ ]:
users_table = users_table.join(users, users.user_id == users_table.user_id, 'left').select(users_table.user_id, users_table.full_name, users_table.email, users_table.phone_number, users_table.address, users_table.address_2, users_table.city, users_table.province, users_table.country, users_table.registered_date,users_table.registered_year, users_table.registered_quarter, users_table.registered_month, users_table.registered_day, users_table.registered_day_name, users_table.registered_hour, users.user_dim_key).sort('registered_date')

In [ ]:
new_users = users_table.filter(users_table.user_dim_key.isNull())
old_users = users_table.filter(users_table.user_dim_key.isNotNull())

---
## Step 5 · Surrogate Key Assignment for New Users

Assigns a unique `user_dim_key` to each new user using `ROW_NUMBER()` over `registered_date` (oldest user gets the lowest key).

The key is offset by `max_value` to avoid collisions with existing records:
```
user_dim_key = ROW_NUMBER() OVER (ORDER BY registered_date) + max_value
```

- On an **initial load** (`flag=0`): `max_value = 0`, so keys start at 1.
- On an **incremental load** (`flag=1`): `max_value = MAX(user_dim_key)` from the existing table, so keys continue seamlessly.

In [ ]:
#  #fetching max_value from existing table
# if flag == '0':
#     max_value = 0
# else:
#     max_value = spark.sql(""" SELECT MAX(user_dim_key) FROM company_db.gold.users_dim """).collect()[0][0]


In [ ]:
max_value = spark.sql(f"""
    SELECT COALESCE(MAX(lot_dim_key), 0) FROM {catalog}.gold.lots_dim
""").collect()[0][0]

In [ ]:
# new_users = new_users.withColumn("user_dim_key", ((F.monotonically_increasing_id()+1)+max_value)).display()
from pyspark.sql import Window
window_spec = Window.orderBy("registered_date")

new_users = new_users.withColumn(
    "user_dim_key",
    (F.row_number().over(window_spec) + F.lit(max_value))
)

In [ ]:
final_users = old_users.union(new_users)

---
## Step 6 · Write to Gold Layer (MERGE)

`old_users` (existing, with keys) and `new_users` (newly keyed) are unioned into `final_users` and registered as a Spark temp view.

The `MERGE INTO company_db.gold.users_dim` then applies two operations atomically:

| Condition | Action | Notes |
|---|---|---|
| `target.user_id = source.user_id` MATCHED | **UPDATE** all fields, set `updated_at = current_timestamp()` | Keeps user contact info in sync |
| NOT MATCHED | **INSERT** new row, set both `created_at` and `updated_at` | Preserves the original insert timestamp |

In [ ]:
final_users.createOrReplaceTempView("final_users")

In [ ]:

spark.sql(
f"""
MERGE INTO {catalog}.gold.users_dim as target
USING final_users as source
ON target.user_id = source.user_id
WHEN MATCHED THEN
UPDATE SET
  target.user_id = source.user_id,
  target.full_name = source.full_name,
  target.phone_number = source.phone_number,
  target.email = source.email,
  target.address = source.address,
  target.address_2 = source.address_2,
  target.city = source.city,
  target.province = source.province,
  target.country = source.country,
  target.registered_date = source.registered_date,
  target.registered_year = source.registered_year,
  target.registered_quarter = source.registered_quarter,
  target.registered_month = source.registered_month,
  target.registered_day = source.registered_day,
  target.registered_hour = source.registered_hour,
  target.registered_day_name = source.registered_day_name,
  target.updated_at = current_timestamp()
WHEN NOT MATCHED THEN INSERT (
  user_dim_key,
  user_id,
  full_name,
  phone_number,
  email,
  address,
  address_2,
  city,
  province,
  country,
  registered_date,
  registered_year,
  registered_quarter,
  registered_month,
  registered_day,
  registered_hour,
  registered_day_name,
  updated_at,
  created_at
)
VALUES
(source.user_dim_key,
source.user_id,
source.full_name,
source.phone_number,
source.email,
source.address,
source.address_2,
source.city,
source.province,
source.country,
source.registered_date,
source.registered_year,
source.registered_quarter,
source.registered_month,
source.registered_day,
source.registered_hour,
source.registered_day_name,
current_timestamp(),
current_timestamp()
)
""")

---
## Step 7 · Verification

Queries the final `company_db.gold.users_dim` table to confirm rows were correctly inserted or updated. Spot-check that:
- `user_dim_key` values are sequential with no gaps for new users
- `created_at` is populated for all rows
- `updated_at` reflects the current run timestamp for modified rows

In [ ]:
# %sql
# DESCRIBE HISTORY company_db.gold.users_dim;

In [ ]:
# %sql
# RESTORE TABLE company_db.gold.users_dim TO VERSION AS OF 6;